In [22]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))  # jeśli notebook jest w podfolderze: Path.cwd().parent / "src"

from src.utils import DataExplorer
import json

import pandas as pd

In [23]:
import sys
print(sys.executable)

/Users/tynka/Documents/pdf-converter/.venv/bin/python


In [24]:
helper = DataExplorer()

In [25]:
from sklearn.metrics import classification_report, f1_score
from src.models.train_and_evaluate import ClassifierConfig, TextClassifier

# 1. Wczytanie danych z podziałem na train i validate (domyślnie 20% unikalnych dokumentów w validate)
train_data, val_data = helper.load_file_jsonl(
    "dataset/dataset_all.jsonl",
    validate_split=0.2,
    shuffle=True,
    random_state=42,
)

print("Train - próbki:", len(train_data["X"]), "| unikalne doc_ids:", len(set(train_data["doc_ids"])))
if val_data is not None:
    print("Validate - próbki:", len(val_data["X"]), "| unikalne doc_ids:", len(set(val_data["doc_ids"])))

# 2. Konfiguracja modelu
simple_config = ClassifierConfig(
    model_type="simple",
    C=0.25,
    max_word_features=2_000,
    word_ngram_range=(1, 1),
    min_df=2,
    max_df=0.95,
    boost_intro_words=50,
    class_weight="balanced",
    random_state=42,
)

# simple_config = ClassifierConfig(
#     model_type="simple",
#     C=0.05,                  # silna regularyzacja L2 (sprawdź też zakres 0.02 - 0.1)
#     max_word_features=800,   # mniejszy słownik = mniej wag do zapamiętania szumu
#     word_ngram_range=(1, 1), # tylko pojedyncze słowa (unigramy)
#     min_df=4,                # słowo musi pojawić się w min. 4 dokumentach
#     max_df=0.80,             # odrzuca bardzo częste słowa w korpusie
#     sublinear_tf=True,       # logarytmiczne TF tłumi dominację powtórzeń słowa
#     boost_intro_words=0,     # brak sztucznego podbijania początku dokumentu (bezpieczniej przy generalizacji)
#     class_weight="balanced",
#     random_state=42,
# )

# simple_config = ClassifierConfig(
#     model_type="simple",
#     C=0.05,                  # nadal mocna regularyzacja L2
#     max_word_features=1_200, # lekko większy limit pod bigramy, ale bez przesady
#     word_ngram_range=(1, 2), # unigramy + bigramy
#     min_df=4,                # bigramy i słowa muszą wystąpić w co najmniej 4 dokumentach
#     max_df=0.80,
#     sublinear_tf=True,
#     boost_intro_words=0,
#     class_weight="balanced",
#     random_state=42,
# )
# 3. Kros-walidacja na zbiorze treningowym (train_data) z zachowaniem grup doc_ids
cv_clf = TextClassifier(config=simple_config)
cv_results = cv_clf.evaluate_cv(
    X=train_data["X"],
    y=train_data["y"],
    groups=train_data["doc_ids"],
    n_splits=5,
    average="macro",
)

print(f"\n[CV na train] Średnie F1 macro po foldach: {cv_results['f1_macro']:.4f} (+/- {cv_results['f1_macro_std']:.4f})")
print("[CV na train] Wyniki F1 macro w kolejnych foldach:", [round(v, 4) for v in cv_results["f1_macro_folds"]])
print("[CV na train] Raport klasyfikacji:\n", cv_results["report"])

# 4. Trenowanie na całym train_data i predykcja / ocena na validation
val_clf = TextClassifier(config=simple_config)
val_clf.fit(train_data["X"], train_data["y"])

if val_data is not None and len(val_data["X"]) > 0:
    val_preds = val_clf.predict(val_data["X"])
    val_f1_macro = f1_score(val_data["y"], val_preds, average="macro", zero_division=0)
    print(f"\n[Validation] F1 macro: {val_f1_macro:.4f}")
    print("[Validation] Raport klasyfikacji:\n", classification_report(val_data["y"], val_preds, zero_division=0))

# 5. Trenowanie finalnego modelu na WSZYSTKICH dostępnych danych (train + validate) i zapis
all_X = train_data["X"] + (val_data["X"] if val_data is not None else [])
all_y = train_data["y"] + (val_data["y"] if val_data is not None else [])

final_clf = TextClassifier(config=simple_config)
final_clf.fit(all_X, all_y)

saved_path = final_clf.save("src/created_models/simple_text_classifier.joblib")
print("\nZapisano finalny model do:", saved_path)

# Opcjonalny test wczytania zapisanego modelu
loaded_clf = TextClassifier.load(saved_path)
print("Model wczytany poprawnie, klasy:", loaded_clf.pipeline.classes_.tolist())

Train - próbki: 237 | unikalne doc_ids: 237
Validate - próbki: 60 | unikalne doc_ids: 60

[CV na train] Średnie F1 macro po foldach: 1.0000 (+/- 0.0000)
[CV na train] Wyniki F1 macro w kolejnych foldach: [1.0, 1.0, 1.0, 1.0, 1.0]
[CV na train] Raport klasyfikacji:
                 precision    recall  f1-score   support

     negatywne       1.00      1.00      1.00        46
     odwolanie       1.00      1.00      1.00        64
pelnomocnictwo       1.00      1.00      1.00        63
       przelew       1.00      1.00      1.00        64

      accuracy                           1.00       237
     macro avg       1.00      1.00      1.00       237
  weighted avg       1.00      1.00      1.00       237


[Validation] F1 macro: 1.0000
[Validation] Raport klasyfikacji:
                 precision    recall  f1-score   support

     negatywne       1.00      1.00      1.00        11
     odwolanie       1.00      1.00      1.00        16
pelnomocnictwo       1.00      1.00      1.00   

In [26]:
from src.models.model_evaluation import _normalize_text, audit_splits, show_top_features, evaluate_confidence_thresholds

In [27]:
audit_splits(train_data, val_data)

# Po wytrenowaniu modelu na train_data (np. val_clf):
show_top_features(val_clf, top_n=10)

=== 1. LICZNOŚCI I UNIKALNE DOKUMENTY ===
Train: 237 rekordów | 237 unikalnych doc_ids
Validate: 60 rekordów | 60 unikalnych doc_ids
Wspólne doc_ids (train & val): 0 -> []

=== 2. ROZKŁAD KLAS ===
Train klasy: {'negatywne': 46, 'odwolanie': 64, 'pelnomocnictwo': 63, 'przelew': 64}
Validate klasy: {'negatywne': 11, 'pelnomocnictwo': 17, 'odwolanie': 16, 'przelew': 16}

=== 3. DUPLIKATY TEKSTÓW ===
Train unikalne znormalizowane teksty: 211 / 237
Validate unikalne znormalizowane teksty: 59 / 60
Identyczne surowe teksty między train a val: 17
Identyczne znormalizowane teksty między train a val: 17
  [DUPLIKAT]: kancelariaradcyprawnegoaleksandrarybak ul.polna115,40 175katowice poznan dzień 11 wrzesnie 2026 prezeskrajowejizbyodwota
  [DUPLIKAT]: ogłoszenie numer token_bzp dzień token_date ogłoszenie zamówić usługa utrzymać ciągły ruch urządzenie sieć wodociągowy s
  [DUPLIKAT]: banknadwislanski spoikaakcyjny ul.bankowa12,00 950warszawa bank nadwislanski www.banknadwislanski.example nazwa fir

In [28]:
df_thresh = evaluate_confidence_thresholds(
    val_clf,
    val_data["X"],
    val_data["y"],
    thresholds=(0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5,0.55, 0.6),
)

print(df_thresh.to_string(index=False))

 threshold  n_samples  coverage_pct  accuracy_pct  f1_macro_pct
      0.20         60        100.00         100.0         100.0
      0.25         60        100.00         100.0         100.0
      0.30         60        100.00         100.0         100.0
      0.35         60        100.00         100.0         100.0
      0.40         60        100.00         100.0         100.0
      0.45         59         98.33         100.0         100.0
      0.50         54         90.00         100.0         100.0
      0.55         45         75.00         100.0         100.0
      0.60         24         40.00         100.0         100.0
